In [1]:
import pandas as pd

# Load the dataset
file_path = 'Cleaned_dataset.csv'
data = pd.read_csv(file_path)

# Display the first few rows and column information for initial exploration
data.head(), data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3300 entries, 0 to 3299
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Type              3300 non-null   object 
 1   Localisation      3300 non-null   object 
 2   Latitude          3300 non-null   float64
 3   Longitude         3300 non-null   float64
 4   Title             3300 non-null   object 
 5   Price in DH       3300 non-null   float64
 6   Tags              3300 non-null   object 
 7   Tags_length       3300 non-null   int64  
 8   Area in m²        3300 non-null   float64
 9   Rooms             3300 non-null   float64
 10  Bedrooms          3300 non-null   float64
 11  Bathrooms         3300 non-null   float64
 12  Condition         3300 non-null   object 
 13  Floor             3300 non-null   float64
 14  House Age in ans  3300 non-null   int64  
dtypes: float64(8), int64(2), object(5)
memory usage: 386.8+ KB


(                      Type                          Localisation   Latitude  \
 0  Appartements Casablanca  Casablanca Finance City à Casablanca  33.561705   
 1  Appartements Casablanca                     Anfa à Casablanca  33.561705   
 2  Appartements Casablanca               La Gironde à Casablanca  33.561705   
 3  Appartements Casablanca          Bourgogne Ouest à Casablanca  33.561705   
 4  Appartements Casablanca           Les princesses à Casablanca  33.574209   
 
    Longitude                                              Title  Price in DH  \
 0  -7.630379      Somptueux appartement à louer au tour végétal      22000.0   
 1  -7.630379     A vendre appartement 86m face ELBILIA lahjajma    1230000.0   
 2  -7.630379        A vendre bel appartement 2 chambres Gironde     900000.0   
 3  -7.630379   A vendre appartement 180m clinique Badr pas cher    2400000.0   
 4  -7.644182  Appartement 3 chambres yaacoub almansour pas cher    1550000.0   
 
                              

In [2]:
# Step 1: Remove unnecessary columns
columns_to_remove = ['Title', 'Tags', 'Tags_length', 'Localisation', 'Latitude', 'Longitude', 'Floor']
data_cleaned = data.drop(columns=columns_to_remove)

# Step 2: Filter rows based on the price range
lower_bound = 50000
upper_bound = 10000000
data_cleaned = data_cleaned[(data_cleaned['Price in DH'] >= lower_bound) & 
                            (data_cleaned['Price in DH'] <= upper_bound)]

# Step 3: Encode categorical variables
data_cleaned = pd.get_dummies(data_cleaned, columns=['Type', 'Condition'], drop_first=True)

# Step 4: Splitting into training (90%) and testing (10%)
from sklearn.model_selection import train_test_split

X = data_cleaned.drop(columns=['Price in DH'])
y = data_cleaned['Price in DH']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

# Displaying the processed data and splits
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((2643, 10), (294, 10), (2643,), (294,))

In [6]:
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense

# Step 5: Normalize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Step 6: Build the Neural Network
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),  # Input layer
    Dense(32, activation='relu'),  # Hidden layer
    Dense(1)  # Output layer for regression
])

# Compile the model
model.compile(optimizer='sgd', loss='mse', metrics=['mae'])

# Display the model summary
model.summary()


2025-01-09 21:03:12.144869: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-01-09 21:03:12.149023: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-01-09 21:03:12.162617: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1736452992.184786  180066 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1736452992.192223  180066 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-01-09 21:03:12.214947: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU ins

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,817 (11.00 KB)

 Trainable params: 2,817 (11.00 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
# Train the model
history = model.fit(X_train_scaled, y_train, 
                    validation_data=(X_test_scaled, y_test), 
                    epochs=100, batch_size=32, verbose=1)

# Evaluate the model on the test data
test_loss, test_mae = model.evaluate(X_test_scaled, y_test, verbose=0)

print(f"Test Loss (MSE): {test_loss:.2f}")
print(f"Test Mean Absolute Error (MAE): {test_mae:.2f}")


Epoch 1/100
83/83 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 1378886593238206909798940672.0000 - mae: 7370706518016.0000 - val_loss: 227653916730230470344704.0000 - val_mae: 477130915840.0000
Epoch 2/100
83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 122711578645198457012224.0000 - mae: 333983285248.0000 - val_loss: 7958023797050158612480.0000 - val_mae: 89207750656.0000
Epoch 3/100
83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 4289575343753944629248.0000 - mae: 62443749376.0000 - val_loss: 278189706335605489664.0000 - val_mae: 16679018496.0000
Epoch 4/100
83/83 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 149949126404091150336.0000 - mae: 11674910720.0000 - val_loss: 9725526693841469440.0000 - val_mae: 3118576896.0000
Epoch 5/100
83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 5242038233654624256.0000 - mae: 2182870528.0000 - val_loss: 340183778032877568.0000 - val_mae: 583247872.0000
Epoch 6/100
83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 183231499644960768.0000 - mae: 408114784.0000 - 

In [10]:
# Example custom input (replace with real values from your dataset)
custom_input = [[120, 5, 3, 2, 7, 1, 0, 0, 0, 1]]  # Replace with actual feature values

# Scale the input
custom_input_scaled = scaler.transform(custom_input)

# Make a prediction
predicted_price = model.predict(custom_input_scaled)
print(f"Predicted Price: {predicted_price[0][0]:.2f} DH")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step
Predicted Price: 2809243.50 DH


/home/am44/anaconda3/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
